In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime  # datetime 모듈 추가
import re

# 인천루키나호


# 크롤링 함수
def crawl_schedule():
    data = []
    
    # 사이트명 추가
    site_name = "루키나호"  # 사이트명을 원하는 대로 설정
    
    # 현재 년도와 월 자동 추출
    current_year = datetime.datetime.now().year  # 현재 년도 가져오기
    current_month = datetime.datetime.now().month  # 현재 월 가져오기
    
    # 12개월 동안 크롤링
    for month in range(current_month, current_month + 12):
        year = current_year
        # 12월을 넘는 경우, 년도를 다음 해로 변경
        if month > 12:
            month = month - 12
            year = current_year + 1
        
        # URL 생성 (2025년부터 시작)
        url = f"https://lukina.sunsang24.com/ship/schedule_fleet/{year}{month:02d}"
        response = requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')
        
        # 날짜 정보 추출
        items = soup.select(".shipsinfo_daywarp")
        
        # 날짜 정보가 없으면 크롤링 종료
        if not items:
            break
        
        # 각 날짜별 정보 추출
        for item in items:
            
            day_part_raw = item.select_one(".date_info").text.strip().replace("\n", "")
            cut_index = day_part_raw.find(")") + 1
            date = day_part_raw[:cut_index]
            
            wave_power = item.select_one(".date_info2").text.strip().replace('\n', '')
            
            # 여러 배 정보 모두 가져오기
            ships = item.select(".small_event_wrap")  # 각 날짜별 배 정보를 가져옴
            for ship in ships:
                ship_name = ship.select_one(".ship_info>.title").text.strip()
                #fish_name = ship.select_one(".fishspecies").text.replace("어종 :", "").split("/")[0].strip()
                try:
                    fish_name = ship.select_one(".fishspecies")
                    if fish_name:
                        fish_name = fish_name.text.replace("어종 :", "").split("/")[0].strip()
                        print([fish_name])  # 리스트 형식으로 출력
                    else:
                        fish_name =[]
                except AttributeError:
                 print([])  # 예외 발생 시 빈 리스트 출력
                
                
            
                # 예약자리 정보 처리: 마감일 경우 '마감'으로 표시
                reservation_element = ship.select_one(".number.blink_me.n_blue.f_20")
                if reservation_element:
                    reservation = reservation_element.text.strip()
                else:
                    reservation = "마감"  # 데이터가 없으면 '마감'으로 표시
                
                # 예약 바로가기 URL 생성
                booking_url = f"https://lukina.sunsang24.com/ship/schedule_fleet/{year}{month:02d}"

                # 모든 배 정보를 data 리스트에 추가
                data.append([site_name, date, wave_power, ship_name, fish_name, reservation, f'=HYPERLINK("{booking_url}", "예약바로가기")'])
    
    # 데이터프레임으로 변환
    df = pd.DataFrame(data, columns=['사이트명', '날짜', '조류세기', '선박명', '어종/낚시방법', '예약자리', '예약바로가기'])
    return df

# 크롤링 실행
df = crawl_schedule()

# 엑셀 파일로 저장 (하이퍼링크 포함)
df.to_excel('result_with_site_name.xlsx', index=False)

# 출력 (원하는 경우)
print(df)


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

# 현재 연도와 월 구하기
current_year = datetime.now().year
current_month = datetime.now().month

site_name = "영종도라이즈호"
base_url = "https://risefishing.sunsang24.com/ship/schedule_fleet"  # 실제 사이트 URL로 변경하세요.

data = []

# 12개월 동안 크롤링
for month in range(current_month, current_month + 12):
    year = current_year
    if month > 12:
        month = month - 12
        year = current_year + 1
    
    # URL 생성
    url = f"{base_url}/{year}{month:02d}"
    response = requests.get(url)
    html = response.text
    soup = BeautifulSoup(html, 'html.parser')
    items = soup.select(".shipsinfo_daywarp")
    
    # 날짜 추출
    for item in items:
        # 선박명
        ship_name = item.select_one(".ship_info>div.title").text.strip()

        # 날짜 및 조류세기
        day_part_raw = item.select_one(".date_wrap").text.strip().replace("\n", "")
        cut_index = day_part_raw.find(")") + 1
        day_part = day_part_raw[:cut_index]
        
        water_part = item.select_one(".date_info2").text.strip()

        # 어종 정보
        fish_name = item.select_one(".fishspecies").text.replace("어종 :", "").split("/")[0].strip()

        # 예약 정보
        reser_img = item.select_one('li.remain').text.strip()
        if "남은자리" in reser_img:
            remaining_number = item.select_one('li.remain span.number').text.strip()
        else:
            remaining_number = "마감"
        
        # 데이터 리스트에 추가
        data.append([site_name, ship_name, day_part, water_part, fish_name, remaining_number])

# DataFrame 생성
df = pd.DataFrame(data, columns=['사이트', '선박명', '날짜', '조류세기', '어종', '예약자리'])



df


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import re

# 현재 연도와 월 구하기
current_year = datetime.now().year
current_month = datetime.now().month

site_name = "영종도오뚜기호"
base_url = "http://ottogi.sunsang24.com/ship/schedule_fleet"  # 실제 사이트 URL로 변경하세요.

data = []

# 12개월 동안 크롤링
for month in range(current_month, current_month + 12):
    year = current_year
    if month > 12:
        month = month - 12
        year = current_year + 1
    
    # URL 생성
    url = f"{base_url}/{year}{month:02d}"
    response = requests.get(url)
    html = response.text
    soup = BeautifulSoup(html, 'html.parser')
    items = soup.select(".shipsinfo_daywarp")
     
    # 날짜 추출
    for item in items:
       # 날짜 및 조류세기
        day_part_raw = item.select_one(".date_wrap").text.strip().replace("\n", "")
        cut_index = day_part_raw.find(")") + 1
        day_part = day_part_raw[:cut_index]
        
        water_part = item.select_one(".date_info2").text.strip()
        
        ships = item.select(".ships_warp>table")
        for ship in ships:
            ship_name = ship.select_one(".ship_info>div.title").text.strip()
            fish_element = ship.select_one("#fish")
            if fish_element is not None:
                fish_name = fish_element.text.strip().replace('[', '').replace(']', '')
            else:
                fish_name = '[]'
            reser_img = ship.select_one('.remain').text.strip()
            if "남은자리" in reser_img:
                remaining_number = ship.select_one('.number.blink_me').text.strip()
            else:
                remaining_number = "마감"
            data.append([site_name, ship_name, day_part, water_part, fish_name, remaining_number])

# DataFrame 생성
df = pd.DataFrame(data, columns=['사이트', '선박명', '날짜', '조류세기', '어종', '예약자리'])
df.to_excel("cruise_schedule.xlsx", index=False, engine='openpyxl')
df
           


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import re

# 현재 연도와 월 구하기
current_year = datetime.now().year
current_month = datetime.now().month


def format_date(raw_date, year):
    try:
        # ✅ 정규식으로 날짜 감지 (연도 포함 또는 미포함)
        match = re.search(r'(\d{1,2})월\s*(\d{1,2})일(\(\w\))?', raw_date)
        if match:
            month = int(match.group(1))
            day = int(match.group(2))
            formatted_date = f"{year}-{month:02d}-{day:02d}"
            return formatted_date
        else:
            print(f"❌ 날짜 형식 인식 실패: {raw_date}")
            return None
    except Exception as e:
        print(f"❌ 날짜 형식 변환 실패: {raw_date} - {e}")
        return None
def crawl_site_supernova(site_name, base_url):
    
    data = []

    # 12개월 동안 크롤링
    for month in range(current_month, current_month + 12):
        year = current_year if month <= 12 else current_year + 1
        month = month if month <= 12 else month - 12
        
        # URL 생성
        url = f"{base_url}/{year}{month:02d}"
        response = requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')
        items = soup.select(".shipsinfo_daywarp")
      
        # 날짜 추출
        for item in items:
        # 날짜 및 조류세기
            
            raw_date = item.select_one(".date_wrap").text.strip().replace("\n", "")
            formatted_date = format_date(raw_date, year)  # ✅ 날짜 변환
            
            if not formatted_date:
                continue  # 날짜 변환 실패 시 무시
            
            
            wave_power = item.select_one(".date_info2").text.strip()
            zone = "인천권"
            
            ships = item.select(".ships_warp>table")
            for ship in ships:
                ship_name = ship.select_one(".ship_info>div.title").text.strip()
                fish_element = ship.select_one("#fish")
                if fish_element is not None:
                    fish_name = fish_element.text.strip().replace('[', '').replace(']', '')
                else:
                    fish_name = '[]'
                reservation = ship.select_one('.remain').text.strip()
                if "남은자리" in reservation:
                    reservation = ship.select_one('.number.blink_me').text.strip()
                else:
                    reservation = "마감"
                booking_url = f"{base_url}/{year}{month:02d}"  
                    
                data.append([zone,site_name, ship_name, formatted_date, wave_power, fish_name, reservation, booking_url])
                
    # DataFrame 생성
    df = pd.DataFrame(data, columns=['지역','사이트', '선박명', '날짜', '조류세기', '어종', '예약자리','바로가기'])
    print(df)
    return data
    
site_data = crawl_site_supernova(site_name, base_url)
    

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import re

# 현재 연도와 월 구하기
current_year = datetime.now().year
current_month = datetime.now().month


def format_date(raw_date, year):
    try:
        # ✅ 정규식으로 날짜 감지 (연도 포함 또는 미포함)
        match = re.search(r'(\d{1,2})월\s*(\d{1,2})일(\(\w\))?', raw_date)
        if match:
            month = int(match.group(1))
            day = int(match.group(2))
            formatted_date = f"{year}-{month:02d}-{day:02d}"
            return formatted_date
        else:
            print(f"❌ 날짜 형식 인식 실패: {raw_date}")
            return None
    except Exception as e:
        print(f"❌ 날짜 형식 변환 실패: {raw_date} - {e}")
        return None
def crawl_site_bigboss(site_name, base_url):
    
    data = []

    # 12개월 동안 크롤링
    for month in range(current_month, current_month + 1):
        year = current_year if month <= 12 else current_year + 1
        month = month if month <= 12 else month - 12
        
        # URL 생성
        url = f"{base_url}/{year}{month:02d}"
        response = requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')
        items = soup.select(".shipsinfo_daywarp")
               # 날짜 추출
        for item in items:
        # 날짜 및 조류세기
            
            raw_date = item.select_one(".date_wrap").text.strip().replace("\n", "")
            formatted_date = format_date(raw_date, year)  # ✅ 날짜 변환
            
            if not formatted_date:
                continue  # 날짜 변환 실패 시 무시
            
            
            wave_power = item.select_one(".date_info2").text.strip()
            zone = "충청권"
            
            ships = item.select(".ships_warp>table")
            for ship in ships:
                ship_name = ship.select_one(".ship_info>div.title").text.strip()
                fish_element = ship.select_one("#fish")
                if fish_element is not None:
                    fish_name = fish_element.text.strip().replace('[', '').replace(']', '')
                else:
                    fish_name = '[]'
                reservation = ship.select_one('.remain').text.strip()
                if "남은자리" in reservation:
                    reservation = ship.select_one('.number.blink_me').text.strip()
                else:
                    reservation = "마감"
                booking_url = f"{base_url}/{year}{month:02d}"  
                    
                data.append([zone,site_name, ship_name, formatted_date, wave_power, fish_name, reservation, booking_url])
                
    # DataFrame 생성
    df = pd.DataFrame(data, columns=['지역','사이트', '선박명', '날짜', '조류세기', '어종', '예약자리','바로가기'])
    print(df)
    return data 
       
           
        
site_name = "오천항빅보스호"
base_url = "https://bigboss24.sunsang24.com/ship/schedule_fleet"         
site_data = crawl_site_bigboss(site_name, base_url)

     지역      사이트   선박명          날짜 조류세기    어종 예약자리  \
0   인천권  오천항빅보스호  빅보스호  2025-05-14   8물  갑오징어   마감   
1   인천권  오천항빅보스호  빅보스호  2025-05-15   9물  갑오징어   마감   
2   인천권  오천항빅보스호  빅보스호  2025-05-16  10물  갑오징어  17명   
3   인천권  오천항빅보스호  빅보스호  2025-05-17  11물  갑오징어  12명   
4   인천권  오천항빅보스호  빅보스호  2025-05-18  12물  갑오징어   8명   
5   인천권  오천항빅보스호  빅보스호  2025-05-19  13물  갑오징어  17명   
6   인천권  오천항빅보스호  빅보스호  2025-05-20   조금  갑오징어  17명   
7   인천권  오천항빅보스호  빅보스호  2025-05-21   무시  갑오징어  17명   
8   인천권  오천항빅보스호  빅보스호  2025-05-22   1물  갑오징어  15명   
9   인천권  오천항빅보스호  빅보스호  2025-05-23   2물  갑오징어  14명   
10  인천권  오천항빅보스호  빅보스호  2025-05-24   3물  갑오징어   마감   
11  인천권  오천항빅보스호  빅보스호  2025-05-25   4물  갑오징어  10명   
12  인천권  오천항빅보스호  빅보스호  2025-05-26   5물  갑오징어  17명   
13  인천권  오천항빅보스호  빅보스호  2025-05-27   7물  갑오징어  17명   
14  인천권  오천항빅보스호  빅보스호  2025-05-28   8물  갑오징어  17명   
15  인천권  오천항빅보스호  빅보스호  2025-05-29   9물  갑오징어  17명   
16  인천권  오천항빅보스호  빅보스호  2025-05-30  10물  갑오징어  17명   
17  인천권  오천항빅보스호  빅보스호  2025

In [ ]:
        # 날짜 추출
        for item in items:
        # 날짜 및 조류세기
            
            raw_date = item.select_one(".date_wrap").text.strip().replace("\n", "")
            formatted_date = format_date(raw_date, year)  # ✅ 날짜 변환
            
            if not formatted_date:
                continue  # 날짜 변환 실패 시 무시
            
            
            wave_power = item.select_one(".date_info2").text.strip()
            zone = "인천권"
            
            ships = item.select(".ships_warp>table")
            for ship in ships:
                ship_name = ship.select_one(".ship_info>div.title").text.strip()
                fish_element = ship.select_one("#fish")
                if fish_element is not None:
                    fish_name = fish_element.text.strip().replace('[', '').replace(']', '')
                else:
                    fish_name = '[]'
                reservation = ship.select_one('.remain').text.strip()
                if "남은자리" in reservation:
                    reservation = ship.select_one('.number.blink_me').text.strip()
                else:
                    reservation = "마감"
                booking_url = f"{base_url}/{year}{month:02d}"  
                    
                data.append([zone,site_name, ship_name, formatted_date, wave_power, fish_name, reservation, booking_url])
                
    # DataFrame 생성
    df = pd.DataFrame(data, columns=['지역','사이트', '선박명', '날짜', '조류세기', '어종', '예약자리','바로가기'])
    print(df)
    return data                             
    
site_data = crawl_site_supernova(site_name, base_url)
    